# Raw overlap versus synthesis normalization

This notebook is the slower companion to `notes/raw-overlap-is-not-the-synthesis-rule.md`.

The question is narrow and practical: quarter-hop framing can look almost flat on the raw overlap sum, but does the weighted overlap-add denominator stay equally calm?


## The two sums

For hop `H`, the earlier sidecar measured the raw overlap profile

```text
s[p] = Σ_k w[p + kH]
```

If analysis and synthesis both use the same window, the weighted overlap-add denominator is instead

```text
d[p] = Σ_k w[p + kH]^2
```

That second quantity is the local gain envelope before any normalization. If it ripples, the reconstruction has to divide by a phase-dependent number.


In [ ]:
import csv
from pathlib import Path

rows = list(csv.DictReader((Path('..') / 'art' / 'window-synthesis-normalization-metrics.csv').open()))
quarter_hop = [row for row in rows if row['hop'] == '32']
[(row['name'], round(float(row['raw_max_deviation_pct']), 4), round(float(row['squared_max_deviation_pct']), 4), round(float(row['synthesis_gain_span_db']), 4)) for row in quarter_hop]


Quarter hop is the revealing middle case. The raw overlap already looks calm for several windows, but the squared overlap tells a less uniform story.

- Hann gets flatter when squared
- Blackman stays usable but no longer looks almost exact
- Blackman-Harris hides a real synthesis bill if you only inspect the raw sum
- flat-top stays expensive even before you remember its ENBW


In [ ]:
from windowlab.overlap import normalized_overlap_add_profile, normalized_squared_overlap_add_profile, normalized_synthesis_gain_profile
from windowlab.windows import build_window

for name in ['hann', 'blackman', 'blackman-harris', 'flattop']:
    window = build_window(name, 128)
    raw = normalized_overlap_add_profile(window, 32)
    squared = normalized_squared_overlap_add_profile(window, 32)
    gain = normalized_synthesis_gain_profile(window, 32)
    print(name)
    print('  raw range    ', round(min(v for _, v in raw), 4), round(max(v for _, v in raw), 4))
    print('  squared range', round(min(v for _, v in squared), 4), round(max(v for _, v in squared), 4))
    print('  gain range   ', round(min(v for _, v in gain), 4), round(max(v for _, v in gain), 4))


## Takeaway

The useful practical split is now sharper:

1. raw overlap flatness tells you whether repeated framing is phase-biased
2. squared overlap flatness tells you whether weighted overlap-add synthesis needs a calm or jumpy normalization term
3. those are related, but they are not interchangeable metrics

That is why the repo's window story now has two framing questions instead of one.
